# **BFCL AST Evaluation & McNemar | MASSIVE-Agents**

---

**Project:** KP Agentic AI + LLM  
**Tahap:** Evaluasi ulang baseline development (Step 2–3)  
**Dataset:** MASSIVE-Agents `en-US` dan `id-ID`  
**Model:** Qwen2.5-7B, Gemma 2 9B, dan Sahabat-AI 9B  
**Output:** BFCL AST Accuracy, internal exact match, uji McNemar, dan kandidat audit fungsi

Notebook ini memakai **prediksi baseline yang sudah tersimpan**, bukan menjalankan ulang inferensi model. Skor AST dihitung dengan **fungsi `ast_checker` dari paket resmi `bfcl-eval`**. Hasil pada subset development ini **bukan** skor leaderboard BFCL dan tidak otomatis sebanding secara langsung dengan angka paper yang menggunakan subset / pengaturan berbeda.

## **1 · Konfigurasi Eksperimen**

---

Seluruh evaluasi dilakukan pada **979 record development** dan **2.937 prediksi** (979 × 3 model). Data `test` tidak digunakan. Versi evaluator dibuat tetap agar hasil dapat direproduksi.

In [1]:
from __future__ import annotations

import csv
import hashlib
import importlib.util
import json
import math
import platform
import subprocess
import sys
from collections import Counter
from datetime import datetime, timezone
from importlib.metadata import PackageNotFoundError, version
from itertools import combinations
from pathlib import Path
from types import SimpleNamespace

from IPython.display import display

PROJECT_ROOT_OVERRIDE = None  # Isi Path(r"D:\KP-Agentic-AI") jika deteksi otomatis gagal.
BFCL_VERSION = "2025.12.17"
AUTO_INSTALL_BFCL = True
EXPECTED_DEVELOPMENT = 979
EXPECTED_PREDICTIONS = 2937
MODELS = [
    "qwen2.5:7b-instruct",
    "gemma2:9b-instruct-q4_K_M",
    "Supa-AI/gemma2-9b-cpt-sahabatai-v1-instruct:q4_k_m",
]
NAMA_MODEL = {
    MODELS[0]: "Qwen2.5-7B",
    MODELS[1]: "Gemma 2 9B",
    MODELS[2]: "Sahabat-AI 9B",
}

print("Python:", sys.version.split()[0])
print("BFCL target version:", BFCL_VERSION)
print("Model:", ", ".join(NAMA_MODEL.values()))

Python: 3.11.9
BFCL target version: 2025.12.17
Model: Qwen2.5-7B, Gemma 2 9B, Sahabat-AI 9B


## **2 · Lokasi Project dan Pemeriksaan File**

---

Dataset dibaca dari `data/processed/massive_agents_reviewed`, sedangkan prediksi lama dibaca dari `results/baseline_evaluation`. Semua output baru disimpan ke `results/bfcl_ast_evaluation` sehingga file eksperimen sebelumnya **tidak ditimpa**.

In [2]:
def locate_project(override=None):
    if override is not None:
        candidates = [Path(override).expanduser()]
    else:
        current = Path.cwd().resolve()
        candidates = [current, *current.parents, Path(r"D:\KP-Agentic-AI")]
    for candidate in candidates:
        data_dir = candidate / "data" / "processed" / "massive_agents_reviewed"
        if data_dir.is_dir():
            return candidate
    raise FileNotFoundError(
        "Folder project belum ditemukan. Atur PROJECT_ROOT_OVERRIDE pada Cell 1."
    )

ROOT = locate_project(PROJECT_ROOT_OVERRIDE)
DATA_DIR = ROOT / "data" / "processed" / "massive_agents_reviewed"
BASELINE_DIR = ROOT / "results" / "baseline_evaluation"
OUT = ROOT / "results" / "bfcl_ast_evaluation"
REPORT_FILE = ROOT / "results" / "preprocessing_reviewed" / "finalization_summary.json"
PREDICTION_FILE = BASELINE_DIR / "predictions_development_full_v1_verified.csv"
DATA_FILES = {
    "en-US": DATA_DIR / "en-US_development.jsonl",
    "id-ID": DATA_DIR / "id-ID_development.jsonl",
}

required_paths = [*DATA_FILES.values(), PREDICTION_FILE, REPORT_FILE]
missing = [p for p in required_paths if not p.is_file()]
if missing:
    raise FileNotFoundError("File belum ditemukan:\n" + "\n".join(map(str, missing)))

OUT.mkdir(parents=True, exist_ok=True)
print("Project:", ROOT)
print("Prediksi lama:", PREDICTION_FILE)
print("Folder output baru:", OUT)

Project: D:\KP-Agentic-AI
Prediksi lama: D:\KP-Agentic-AI\results\baseline_evaluation\predictions_development_full_v1_verified.csv
Folder output baru: D:\KP-Agentic-AI\results\bfcl_ast_evaluation


## **3 · Membaca Dataset dan Prediksi Baseline**

---

Pemeriksaan meliputi jumlah record, `row_key` unik, label bahasa, `group_id`, split, nama fungsi target, katalog 55 fungsi, serta kelengkapan prediksi untuk setiap model. Ketidaksesuaian menghentikan proses sebelum evaluasi.

In [3]:
def read_jsonl(path):
    rows = []
    with path.open("r", encoding="utf-8-sig") as file:
        for line_no, line in enumerate(file, 1):
            if not line.strip():
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"JSONL rusak: {path.name}:{line_no}") from exc
    return rows


def read_csv(path):
    with path.open("r", encoding="utf-8-sig", newline="") as file:
        return list(csv.DictReader(file))


def as_bool(value):
    return value is True or str(value).strip().casefold() == "true"


preprocessing_summary = json.loads(REPORT_FILE.read_text(encoding="utf-8-sig"))
if preprocessing_summary.get("anti_leakage_group_and_id") != "PASS":
    raise ValueError("Status anti-data-leakage belum PASS.")
if preprocessing_summary.get("gt_modified") is not False:
    raise ValueError("Ground truth sebelumnya terindikasi berubah; periksa sumber.")

records = {}
lang_counts = Counter()
for language, filepath in DATA_FILES.items():
    for record in read_jsonl(filepath):
        key = record["row_key"]
        assert key not in records, f"Duplikat row_key: {key}"
        assert record["split"] == "development", key
        assert record["language_code"] == language, key
        assert record["test_category"] == "multiple_function", key
        assert isinstance(record["function"], list) and len(record["function"]) == 55, key
        assert isinstance(record["gt"], dict) and len(record["gt"]) == 1, key
        function_names = [item["name"] for item in record["function"]]
        assert len(set(function_names)) == 55, key
        assert next(iter(record["gt"])) in function_names, key
        records[key] = record
        lang_counts[language] += 1

assert len(records) == EXPECTED_DEVELOPMENT == preprocessing_summary["development"]
assert lang_counts == {"en-US": 582, "id-ID": 397}

predictions = read_csv(PREDICTION_FILE)
assert len(predictions) == EXPECTED_PREDICTIONS
assert {p["model"] for p in predictions} == set(MODELS)
assert len({p["fingerprint"] for p in predictions}) == 1
seen = set()
per_model = Counter()
for p in predictions:
    pair = (p["model"], p["row_key"])
    assert pair not in seen, f"Duplikat prediksi: {pair}"
    seen.add(pair)
    assert p["row_key"] in records, pair
    r = records[p["row_key"]]
    assert r["language_code"] == p["language_code"], pair
    assert r["group_id"] == p["group_id"], pair
    assert str(r["source_record_id"]) == str(p["record_id"]), pair
    assert next(iter(r["gt"])) == p["target_function"], pair
    per_model[p["model"]] += 1
assert set(per_model) == set(MODELS)
assert all(n == EXPECTED_DEVELOPMENT for n in per_model.values())

print("Pemeriksaan data: PASS")
print("Development:", len(records), dict(lang_counts))
print("Prediksi:", len(predictions), dict(per_model))
print("Audit preprocessing:", preprocessing_summary.get("audit_status", "tidak tersedia"))

Pemeriksaan data: PASS
Development: 979 {'en-US': 582, 'id-ID': 397}
Prediksi: 2937 {'qwen2.5:7b-instruct': 979, 'gemma2:9b-instruct-q4_K_M': 979, 'Supa-AI/gemma2-9b-cpt-sahabatai-v1-instruct:q4_k_m': 979}
Audit preprocessing: provisional decisions; methodological/source verification still required


## **4 · Memuat Evaluator AST Resmi BFCL**

---

Menggunakan paket `bfcl-eval` versi tetap dan memanggil `bfcl_eval.eval_checker.ast_eval.ast_checker.ast_checker` secara langsung. Ini **bukan** implementasi AST buatan sendiri. Nama fungsi yang mengandung titik (`calendar.set`, dsb.) tetap dipertahankan melalui pengaturan alias model yang diperlukan evaluator.

**Jika instalasi gagal**, periksa koneksi internet dan versi Python, lalu kirim traceback cell ini. Jangan mengubah logika evaluator atau GT.

In [4]:
%pip install bfcl-eval==2025.12.17

Note: you may need to restart the kernel to use updated packages.


In [5]:
import sys
import subprocess

print("Python:", sys.version)
print("Environment:", sys.executable)

result = subprocess.run(
    [
        sys.executable, "-m", "pip",
        "install", "bfcl-eval==2025.12.17"
    ],
    capture_output=True,
    text=True
)

print("\n=== HASIL INSTALASI ===")
print("Return code:", result.returncode)
print("\n".join((result.stdout + "\n" + result.stderr).splitlines()[-50:]))

Python: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
Environment: d:\KP-Agentic-AI\.venv-bfcl\Scripts\python.exe

=== HASIL INSTALASI ===
Return code: 0



In [6]:
def installed_bfcl_version():
    try:
        return version("bfcl-eval")
    except PackageNotFoundError:
        return None

current_version = installed_bfcl_version()
if current_version != BFCL_VERSION:
    if not AUTO_INSTALL_BFCL:
        raise RuntimeError(
            f"bfcl-eval versi {BFCL_VERSION} diperlukan, terpasang: {current_version}. "
            f"Jalankan %pip install bfcl-eval=={BFCL_VERSION}"
        )
    print(f"Memasang bfcl-eval=={BFCL_VERSION} pada kernel saat ini ...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", f"bfcl-eval=={BFCL_VERSION}"
    ])
    importlib.invalidate_caches()

if installed_bfcl_version() != BFCL_VERSION:
    raise RuntimeError("Versi BFCL belum sesuai. Restart kernel, lalu jalankan ulang.")

try:
    from bfcl_eval.eval_checker.ast_eval.ast_checker import ast_checker
    from bfcl_eval.constants.enums import Language
    from bfcl_eval.constants.model_config import MODEL_CONFIG_MAPPING
except Exception as exc:
    raise RuntimeError(
        "Import checker resmi BFCL gagal. Coba Restart Kernel lalu Run All. "
        "Jika tetap gagal, kirim traceback kepada saya."
    ) from exc

for model in MODELS:
    # BFCL memanggil model_name.replace('_', '/') pada convert_func_name.
    MODEL_CONFIG_MAPPING[model] = SimpleNamespace(underscore_to_dot=False)
    MODEL_CONFIG_MAPPING[model.replace("_", "/")] = SimpleNamespace(underscore_to_dot=False)

print("Official AST checker imported:", ast_checker.__module__)
print("Versi bfcl-eval:", installed_bfcl_version())

Official AST checker imported: bfcl_eval.eval_checker.ast_eval.ast_checker
Versi bfcl-eval: 2025.12.17


## **5 · Evaluasi AST Per Prediksi**

---

Prediksi JSON yang tersimpan diubah menjadi format input resmi AST checker: `[{nama_fungsi: argumen}]`. Ground truth tetap berbentuk daftar alternatif yang disediakan dataset. Prediksi invalid JSON tetap salah, sedangkan **error integrasi evaluator harus menghentikan proses**, bukan otomatis dihitung sebagai kesalahan model.

In [7]:
def evaluate_one(prediction, record):
    if not as_bool(prediction["parsed_json"]):
        return False, "adapter:invalid_json", "Output model tidak lolos parser JSON baseline"

    function_name = prediction["prediction_name"]
    try:
        arguments = json.loads(prediction["prediction_arguments_json"])
    except json.JSONDecodeError:
        return False, "adapter:invalid_arguments_json", "Kolom argumen tidak valid JSON"

    if not function_name or not isinstance(arguments, dict):
        return False, "adapter:invalid_structure", "Nama fungsi kosong / argumen bukan objek JSON"

    try:
        answer = ast_checker(
            func_description=record["function"],
            model_output=[{function_name: arguments}],
            possible_answer=[record["gt"]],
            language=Language.PYTHON,
            test_category="multiple_function",
            model_name=prediction["model"],
        )
    except Exception as exc:
        raise RuntimeError(
            f"Checker resmi gagal pada {prediction['model']} | {prediction['row_key']}"
        ) from exc

    if not isinstance(answer, dict) or not isinstance(answer.get("valid"), bool):
        raise RuntimeError(f"Format respons AST checker tidak dikenali: {answer!r}")

    return answer["valid"], (
        "" if answer["valid"] else str(answer.get("error_type", "unknown_checker_error"))
    ), json.dumps(answer.get("error", []), ensure_ascii=False)


results = []
for no, prediction in enumerate(predictions, 1):
    record = records[prediction["row_key"]]
    ast_ok, error_type, error_message = evaluate_one(prediction, record)
    results.append({
        "model": prediction["model"],
        "row_key": prediction["row_key"],
        "record_id": prediction["record_id"],
        "language_code": prediction["language_code"],
        "group_id": prediction["group_id"],
        "question": record["question"],
        "target_function": prediction["target_function"],
        "prediction_function": prediction["prediction_name"],
        "prediction_arguments_json": prediction["prediction_arguments_json"],
        "ground_truth_json": json.dumps(record["gt"], ensure_ascii=False),
        "function_correct_internal": as_bool(prediction["function_correct"]),
        "full_match_internal": as_bool(prediction["full_match"]),
        "bfcl_ast_correct": ast_ok,
        "ast_error_type": error_type,
        "ast_error_message": error_message,
        "previous_status": prediction["status"],
    })

assert len(results) == EXPECTED_PREDICTIONS
assert len({(r["model"], r["row_key"]) for r in results}) == EXPECTED_PREDICTIONS
print("AST checker selesai:", len(results), "prediksi")
print("AST benar:", sum(r["bfcl_ast_correct"] for r in results))
print("Jenis kesalahan teratas:", Counter(r["ast_error_type"] for r in results if not r["bfcl_ast_correct"]).most_common(8))

AST checker selesai: 2937 prediksi
AST benar: 716
Jenis kesalahan teratas: [('value_error:string', 1235), ('simple_function_checker:wrong_func_name', 520), ('type_error:simple', 393), ('simple_function_checker:missing_optional', 63), ('simple_function_checker:missing_required', 5), ('simple_function_checker:unexpected_param', 4), ('adapter:invalid_json', 1)]


## **6 · Ringkasan AST vs Exact Match Internal**

---

AST Accuracy menjadi metrik utama. Exact match internal dan ketepatan fungsi dipertahankan sebagai metrik pelengkap. Ringkasan dibuat per model, per bahasa, dan gabungan (`ALL`) tanpa mencampur test set.

In [8]:
def summarize_results(rows):
    output = []
    for model in MODELS:
        for language in ("en-US", "id-ID", "ALL"):
            group = [
                r for r in rows
                if r["model"] == model and (language == "ALL" or r["language_code"] == language)
            ]
            n = len(group)
            ast_n = sum(r["bfcl_ast_correct"] for r in group)
            em_n = sum(r["full_match_internal"] for r in group)
            func_n = sum(r["function_correct_internal"] for r in group)
            output.append({
                "model": model,
                "nama_model": NAMA_MODEL[model],
                "language_code": language,
                "n": n,
                "ast_correct": ast_n,
                "ast_accuracy": round(ast_n / n, 6),
                "internal_exact_correct": em_n,
                "internal_exact_accuracy": round(em_n / n, 6),
                "function_correct": func_n,
                "function_accuracy": round(func_n / n, 6),
                "ast_minus_internal_pp": round(100 * (ast_n - em_n) / n, 4),
            })
    return output

summary_rows = summarize_results(results)
try:
    import pandas as pd
    display(pd.DataFrame(summary_rows)[[
        "nama_model", "language_code", "n", "ast_accuracy",
        "internal_exact_accuracy", "function_accuracy", "ast_minus_internal_pp"
    ]])
except ImportError:
    for item in summary_rows: print(item)

print("Catatan: perbedaan AST vs internal exact dipengaruhi aturan penilaian, bukan inferensi baru.")

,nama_model,language_code,n,ast_accuracy,internal_exact_accuracy,function_accuracy,ast_minus_internal_pp
0,Qwen2.5-7B,en-US,582,0.300687,0.292096,0.831615,0.8591
1,Qwen2.5-7B,id-ID,397,0.219144,0.214106,0.735516,0.5038
2,Qwen2.5-7B,ALL,979,0.267620,0.260470,0.792646,0.7150
3,Gemma 2 9B,en-US,582,0.230241,0.226804,0.862543,0.3436
4,Gemma 2 9B,id-ID,397,0.176322,0.173804,0.785894,0.2519
5,Gemma 2 9B,ALL,979,0.208376,0.205312,0.831461,0.3064
6,Sahabat-AI 9B,en-US,582,0.266323,0.262887,0.862543,0.3436
7,Sahabat-AI 9B,id-ID,397,0.239295,0.231738,0.816121,0.7557
8,Sahabat-AI 9B,ALL,979,0.255363,0.250255,0.843718,0.5107


Catatan: perbedaan AST vs internal exact dipengaruhi aturan penilaian, bukan inferensi baru.


## **7 · Uji Statistik McNemar**

---

McNemar membandingkan dua model **pada record yang sama**, berdasarkan status benar/salah AST. Digunakan uji binomial dua sisi *exact McNemar*, kemudian koreksi **Holm–Bonferroni** untuk 3 perbandingan antarmodel per metrik. Uji internal exact match ditampilkan terpisah sebagai analisis tambahan.

**Catatan metodologis:** sebagian EN dan ID adalah instruksi berpasangan sehingga observasi antarbahasa mungkin berkorelasi. Hasil gabungan dibaca bersama uji per bahasa; untuk kesimpulan akhir dapat ditambah analisis berkelompok berdasarkan `group_id`.

In [9]:
def mcnemar_exact_pvalue(b, c):
    n = b + c
    if n == 0:
        return 1.0
    k = min(b, c)
    log_terms = [
        math.lgamma(n + 1) - math.lgamma(i + 1) - math.lgamma(n - i + 1) - n * math.log(2)
        for i in range(k + 1)
    ]
    m = max(log_terms)
    return min(1.0, 2 * math.exp(m) * sum(math.exp(x - m) for x in log_terms))


def mcnemar_table(rows, metric, language="ALL"):
    filtered = [r for r in rows if language == "ALL" or r["language_code"] == language]
    scores = {
        model: {r["row_key"]: bool(r[metric]) for r in filtered if r["model"] == model}
        for model in MODELS
    }
    all_keys = set(scores[MODELS[0]])
    assert all(set(scores[m]) == all_keys for m in MODELS), "McNemar tidak berpasangan"
    n = len(all_keys)
    output = []
    for model_a, model_b in combinations(MODELS, 2):
        a, b = scores[model_a], scores[model_b]
        both_correct = sum(a[k] and b[k] for k in all_keys)
        a_only = sum(a[k] and not b[k] for k in all_keys)
        b_only = sum(not a[k] and b[k] for k in all_keys)
        both_wrong = n - both_correct - a_only - b_only
        output.append({
            "metric": metric,
            "language_code": language,
            "model_a": model_a,
            "model_b": model_b,
            "n_paired": n,
            "both_correct": both_correct,
            "a_correct_b_wrong": a_only,
            "a_wrong_b_correct": b_only,
            "both_wrong": both_wrong,
            "accuracy_a": round((both_correct + a_only) / n, 6),
            "accuracy_b": round((both_correct + b_only) / n, 6),
            "difference_a_minus_b_pp": round(100 * (a_only - b_only) / n, 4),
            "p_mcnemar_exact": mcnemar_exact_pvalue(a_only, b_only),
        })
    # Koreksi Holm hanya di dalam keluarga 3 perbandingan ini.
    order = sorted(range(len(output)), key=lambda i: output[i]["p_mcnemar_exact"])
    previous_adjusted = 0.0
    for rank, i in enumerate(order):
        adjusted = min(1.0, (len(order) - rank) * output[i]["p_mcnemar_exact"])
        previous_adjusted = max(previous_adjusted, adjusted)
        output[i]["p_holm"] = previous_adjusted
        output[i]["significant_0_05"] = previous_adjusted < 0.05
    return output


mcnemar_rows = []
for metric in ("bfcl_ast_correct", "full_match_internal"):
    for language in ("ALL", "en-US", "id-ID"):
        mcnemar_rows.extend(mcnemar_table(results, metric, language))

try:
    import pandas as pd
    display(pd.DataFrame([
        r for r in mcnemar_rows
        if r["metric"] == "bfcl_ast_correct" and r["language_code"] == "ALL"
    ])[ ["model_a", "model_b", "a_correct_b_wrong", "a_wrong_b_correct",
        "difference_a_minus_b_pp", "p_mcnemar_exact", "p_holm", "significant_0_05"] ])
except ImportError:
    for item in mcnemar_rows[:3]: print(item)

print("Uji utama: AST Accuracy | Uji tambahan: internal exact match")

,model_a,model_b,a_correct_b_wrong,a_wrong_b_correct,difference_a_minus_b_pp,p_mcnemar_exact,p_holm,significant_0_05
0,qwen2.5:7b-instruct,gemma2:9b-instruct-q4_K_M,142,84,5.9244,0.000139,0.000416,True
1,qwen2.5:7b-instruct,Supa-AI/gemma2-9b-cpt-sahabatai-v1-instruct:q4...,126,114,1.2257,0.477752,0.477752,False
2,gemma2:9b-instruct-q4_K_M,Supa-AI/gemma2-9b-cpt-sahabatai-v1-instruct:q4...,56,102,-4.6987,0.000315,0.000629,True


Uji utama: AST Accuracy | Uji tambahan: internal exact match


## **8 · Kandidat Audit Kesalahan Fungsi**

---

Kasus `wrong_function` dari baseline diekspor agar dapat ditinjau apakah benar model salah atau ground truth ambigu. Kolom tinjauan manual sengaja dibiarkan kosong; notebook **tidak menganggap GT salah secara otomatis**.

In [10]:
audit_rows = []
for row in results:
    if row["previous_status"] != "wrong_function":
        continue
    audit_rows.append({
        "model": row["model"],
        "row_key": row["row_key"],
        "language_code": row["language_code"],
        "group_id": row["group_id"],
        "question": row["question"],
        "target_function": row["target_function"],
        "prediction_function": row["prediction_function"],
        "prediction_arguments_json": row["prediction_arguments_json"],
        "ground_truth_json": row["ground_truth_json"],
        "manual_label": "BELUM_DIREVIEW",
        "alasan": "",
        "sumber_verifikasi": "",
    })
print("Kandidat salah fungsi untuk audit manual:", len(audit_rows))
print("Per model:", dict(Counter(r["model"] for r in audit_rows)))

Kandidat salah fungsi untuk audit manual: 520
Per model: {'qwen2.5:7b-instruct': 203, 'gemma2:9b-instruct-q4_K_M': 165, 'Supa-AI/gemma2-9b-cpt-sahabatai-v1-instruct:q4_k_m': 152}


## **9 · Simpan Hasil Evaluasi**

---

CSV evaluasi per prediksi, ringkasan metrik, hasil McNemar, kandidat audit salah fungsi, dan metadata versi disimpan di folder output baru. Ground truth maupun file prediksi asli tidak diubah.

In [11]:
def write_csv(path, rows):
    if not rows:
        raise ValueError(f"Tidak ada data untuk {path.name}")
    with path.open("w", encoding="utf-8-sig", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=list(rows[0]), extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)


def sha256_file(path, buffer_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for part in iter(lambda: stream.read(buffer_size), b""):
            digest.update(part)
    return digest.hexdigest()


output_files = {
    "detail": OUT / "ast_per_prediction.csv",
    "summary": OUT / "summary_ast_vs_internal.csv",
    "mcnemar": OUT / "mcnemar_perbandingan_model.csv",
    "audit": OUT / "audit_salah_fungsi.csv",
    "metadata": OUT / "metadata_run.json",
    "report": OUT / "RINGKASAN_HASIL.md",
}
write_csv(output_files["detail"], results)
write_csv(output_files["summary"], summary_rows)
write_csv(output_files["mcnemar"], mcnemar_rows)
write_csv(output_files["audit"], audit_rows)

metadata = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "python_version": platform.python_version(),
    "bfcl_eval_version": installed_bfcl_version(),
    "bfcl_checker": ast_checker.__module__ + "." + ast_checker.__name__,
    "test_category": "multiple_function",
    "language_enum": "Language.PYTHON",
    "split": "development",
    "dataset_rows": len(records),
    "prediction_rows": len(predictions),
    "models": MODELS,
    "dataset_audit_status": preprocessing_summary.get("audit_status"),
    "prediction_file_sha256": sha256_file(PREDICTION_FILE),
    "dataset_files_sha256": {lang: sha256_file(path) for lang, path in DATA_FILES.items()},
    "notes": [
        "Evaluation uses official BFCL AST checker on custom MASSIVE-Agents development subset, not official full BFCL CLI leaderboard.",
        "Original ground truth and predictions were not modified.",
        "Same 979 development records used for each model.",
        "Internal exact match reused from previous baseline CSV.",
        "McNemar uses 3 paired comparisons per family and Holm correction; related EN-ID records may violate independence.",
        "Provisional GT/audit review still pending.",
    ],
}
output_files["metadata"].write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")

lines = [
    "# Hasil Evaluasi BFCL AST — MASSIVE-Agents (Development)", "",
    f"- Checker: `{metadata['bfcl_checker']}` | `bfcl-eval=={BFCL_VERSION}`",
    f"- Dataset: {len(records)} record (582 EN, 397 ID) | {len(predictions)} prediksi",
    "- Skor di bawah ini berasal dari subset development, bukan skor leaderboard resmi.",
    "- Audit GT dan kasus ambigu masih perlu verifikasi manual.", "",
    "## Perbandingan metrik", "",
    "| Model | Bahasa | N | AST (%) | Internal exact (%) | Fungsi (%) |",
    "|---|---|---:|---:|---:|---:|",
]
for r in summary_rows:
    lines.append(
        f"| {r['nama_model']} | {r['language_code']} | {r['n']} | "
        f"{100*r['ast_accuracy']:.2f} | {100*r['internal_exact_accuracy']:.2f} | "
        f"{100*r['function_accuracy']:.2f} |"
    )
lines.extend([
    "", "## Exact McNemar + Holm (AST, gabungan)", "",
    "| Model A | Model B | A benar/B salah | A salah/B benar | Selisih (pp) | p exact | p Holm |",
    "|---|---|---:|---:|---:|---:|---:|",
])
for r in mcnemar_rows:
    if r["metric"] == "bfcl_ast_correct" and r["language_code"] == "ALL":
        lines.append(
            f"| {NAMA_MODEL[r['model_a']]} | {NAMA_MODEL[r['model_b']]} | "
            f"{r['a_correct_b_wrong']} | {r['a_wrong_b_correct']} | "
            f"{r['difference_a_minus_b_pp']:.2f} | {r['p_mcnemar_exact']:.4g} | {r['p_holm']:.4g} |"
        )
lines.extend([
    "", "## Catatan metodologis", "",
    "AST dinilai menggunakan fungsi checker resmi BFCL, tetapi dataset adalah subset development",
    "MASSIVE-Agents dan inferensi memakai prompt lokal. Karena itu jangan menyamakan skor secara",
    "langsung dengan angka hasil paper tanpa menyesuaikan kondisi eksperimen.", "",
    "McNemar membandingkan record yang sama. Pasangan EN-ID berpotensi berkorelasi;",
    "perbandingan per bahasa disertakan dan pemeriksaan berbasis group_id direkomendasikan.", "",
    "Hasil audit manual tetap provisional dan tidak otomatis diperbaiki oleh checker.",
])
output_files["report"].write_text("\n".join(lines) + "\n", encoding="utf-8")

print("SELESAI! Semua output disimpan di:", OUT)
for name, path in output_files.items():
    print(f"- {name}: {path.name}")

SELESAI! Semua output disimpan di: D:\KP-Agentic-AI\results\bfcl_ast_evaluation
- detail: ast_per_prediction.csv
- summary: summary_ast_vs_internal.csv
- mcnemar: mcnemar_perbandingan_model.csv
- audit: audit_salah_fungsi.csv
- metadata: metadata_run.json
- report: RINGKASAN_HASIL.md


## **10 · Interpretasi dan Langkah Berikutnya**

---

- **BFCL AST** adalah metrik utama penelitian sesuai revisi mentor. **Internal exact match** tetap dilaporkan sebagai tambahan.
- **McNemar** menilai apakah perbedaan prediksi benar/salah antarmodel konsisten pada record yang sama; perhatikan nilai `p_holm`, arah selisih, dan keterbatasan ketergantungan EN–ID.
- **Audit ambiguitas GT** tidak otomatis selesai hanya karena checker berhasil; kasus perlu ditinjau terhadap instruksi asli, fungsi alternatif, dan skema.
- **Lanjut tahap berikutnya:** periksa laporan ini, audit GT ambigu, lalu pilih satu model tool-calling baru dan rancang eksperimen bahasa lokal.

**Yang dikirim kembali ke chat:** file `RINGKASAN_HASIL.md`, `summary_ast_vs_internal.csv`, `mcnemar_perbandingan_model.csv`, `metadata_run.json`, dan bila memungkinkan `ast_per_prediction.csv` dari `D:\KP-Agentic-AI\results\bfcl_ast_evaluation`.

**Referensi:** [BFCL GitHub](https://github.com/ShishirPatil/gorilla/tree/main/berkeley-function-call-leaderboard) · [MASSIVE-Agents (EMNLP 2025)](https://aclanthology.org/2025.findings-emnlp.1099/)